In [3]:
import os
import requests
import pandas as pd
import mysql.connector
from datetime import datetime, timedelta
from pytz import timezone
from dotenv import load_dotenv

In [4]:
from datetime import datetime
from getpass import getpass

In [6]:
# Load environment variables
load_dotenv()

# Get API key from .env file
api_key = os.getenv("RAPIDAPI_KEY")

# Define parameters (Targeting TOMORROW's flights)
icao = "EDDB"  # Berlin Brandenburg Airport
tomorrow = datetime.now().date() + timedelta(days=1)
time_1 = "00:00"
time_2 = "11:59"

# Construct the URL with 'tomorrow' instead of today's date
url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time_1}/{tomorrow}T{time_2}"

# Query string parameters to filter flight data
querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false"
}

# Request headers for RapidAPI authentication
headers = {
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
    'x-rapidapi-key': api_key
}

# Send the API request and parse the JSON response
response = requests.get(url, headers=headers, params=querystring)

if response.status_code == 200:
    flights_json = response.json()
    print("Flight data retrieved successfully!")
else:
    print(f"Error! Status Code: {response.status_code}")
    print(response.text)

Flight data retrieved successfully!


In [7]:
flight_items = []

for item in flights_json["arrivals"]:
  flight_item = {
      "arrival_airport_icao": icao,
      "departure_airport_icao": item["departure"]["airport"].get("icao", None),
      "scheduled_arrival_time": item["arrival"]["scheduledTime"].get("local", None),
      "flight_number": item.get("number", None)
  }

  flight_items.append(flight_item)

flights_df = pd.DataFrame(flight_items)

flights_df.head()

,arrival_airport_icao,departure_airport_icao,scheduled_arrival_time,flight_number
0,EDDB,NaN,2026-09-29 05:00+02:00,WT 4006
1,EDDB,LTBJ,2026-09-29 06:05+02:00,XQ 966
2,EDDB,LTAJ,2026-09-29 06:45+02:00,XQ 1766
3,EDDB,LROP,2026-09-29 07:00+02:00,W4 3109
4,EDDB,KEWR,2026-09-29 07:15+02:00,UA 962


In [ ]:
# The next step is to pull the city info from our sql database 
# to automate for all three cities we are working on


# Load environment variables
load_dotenv()

# Connecting to Gans MySQL Database
db_connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="Gans"
)

# Integrating sql code by creating a cursor
cursor = db_connection.cursor(buffered=True)
cursor.execute("SELECT city_name FROM cities;")

# Creating a list
city_list = [row[0] for row in cursor.fetchall()]

# Display the list of cities fetched from MySQL
print("Cities fetched from MySQL database:", city_list)

Cities fetched from MySQL database: ['Berlin', 'Hamburg', 'Munich']


In [ ]:
# Creating a new function using another api connection to pull 
# the airport codes by using the cities we pulled from the database previously

def get_airport_icaos_by_cities(city_list, api_key):
    
    icao_list = []
    
    headers = {
        "X-RapidAPI-Key": api_key,
        "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
    }

    for city in city_list:
        url = "https://aerodatabox.p.rapidapi.com/airports/search/term"
        querystring = {"q": city, "limit": "1"}

        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
           
            # Here we are trying to make see if we get the right data if not what is the problem
            
            if "items" in data and len(data["items"]) > 0:
                icao_code = data["items"][0].get("icao")
                if icao_code:
                    icao_list.append(icao_code)
                    print(f"Mapped city '{city}' -> ICAO: {icao_code}")
                else:
                    print(f"No ICAO code found for city '{city}'.")
            else:
                print(f"No airport found matching city '{city}'.")
        else:
            print(f"Failed API search for '{city}'. Status code: {response.status_code}")

    return icao_list


api_key = os.getenv("RAPIDAPI_KEY")

icao_list = get_airport_icaos_by_cities(city_list, api_key)

print("\nGenerated ICAO List:", icao_list)

Mapped city 'Berlin' -> ICAO: EDDB
Mapped city 'Hamburg' -> ICAO: EDDH
Mapped city 'Munich' -> ICAO: EDDM

Generated ICAO List: ['EDDB', 'EDDH', 'EDDM']


In [ ]:
# Create airports table in Gans database and insert target airport details using Python.

load_dotenv()

# Connect to Gans database
db_connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="Gans"
)

cursor = db_connection.cursor()

# 1. Create airports table
create_table_query = """
CREATE TABLE IF NOT EXISTS airports (
    airport_icao VARCHAR(10) PRIMARY KEY,
    airport_name VARCHAR(100),
    city_id INT,
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""
cursor.execute(create_table_query)

# 2. Insert Gans target airports
insert_airports_query = """
INSERT INTO airports (airport_icao, airport_name, city_id)
VALUES 
    ('EDDB', 'Berlin Brandenburg Airport', 1),
    ('EDDH', 'Hamburg Airport', 2),
    ('EDDM', 'Munich Airport', 3)
ON DUPLICATE KEY UPDATE airport_name=VALUES(airport_name), city_id=VALUES(city_id);
"""
cursor.execute(insert_airports_query)

# Commit changes and close connection
db_connection.commit()
cursor.close()
db_connection.close()

print("Airports table created and records inserted successfully in Gans database!")

Airports table created and records inserted successfully in Gans database!


In [ ]:
# This step will be about combining the flight info with the cities 
# we are working on by creating a function
# that will take the flight info just like the first api call we wrote
# and apply it to the cities we have using for loop


def tomorrows_flight_arrivals(icao_list, api_key):
    
    # selecting the next day as our target
 
    berlin_timezone = timezone('Europe/Berlin')
    today = datetime.now(berlin_timezone).date()
    tomorrow = today + timedelta(days=1)

    list_for_arrivals_df = []

    # Looking at each airport ICAO code
    for icao in icao_list:
        # spliting the day as the api works
        times = [["00:00", "11:59"], ["12:00", "23:59"]]

        for time_slot in times:
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time_slot[0]}/{tomorrow}T{time_slot[1]}"
            querystring = {"direction": "Arrival", "withCancelled": "false"}

            headers = {
                "X-RapidAPI-Key": api_key,
                "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
            }

            # Send API request
            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:
                flights_resp = response.json()

                # Just like the first df above we normalize JSON flight data into a DataFrame
                if "arrivals" in flights_resp and flights_resp["arrivals"]:
                    arrivals_df = pd.json_normalize(flights_resp["arrivals"])[
                        [
                            "number", 
                            "airline.name", 
                            "movement.scheduledTime.local", 
                            "movement.terminal", 
                            "movement.airport.name", 
                            "movement.airport.icao"
                        ]
                    ]

                    # Rename columns
                    arrivals_df = arrivals_df.rename(columns={
                        "number": "flight_number", 
                        "airline.name": "airline", 
                        "movement.scheduledTime.local": "arrival_time", 
                        "movement.terminal": "arrival_terminal", 
                        "movement.airport.name": "departure_city", 
                        "movement.airport.icao": "departure_airport_icao"
                    })

                    

                    # Assigning the columns
                    arrivals_df["arrival_airport_icao"] = icao
                    arrivals_df["data_retrieved_on"] = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")

                    # Reordering columns for a better visiblility
                    arrivals_df = arrivals_df[[
                        "arrival_airport_icao", 
                        "flight_number", 
                        "airline", 
                        "arrival_time", 
                        "arrival_terminal", 
                        "departure_city", 
                        "departure_airport_icao", 
                        "data_retrieved_on"
                    ]]

                    # Clean arrival time string format
                    arrivals_df["arrival_time"] = arrivals_df["arrival_time"].str.split("+").str[0]

                    list_for_arrivals_df.append(arrivals_df)
            else:
                print(f"Failed to fetch data for {icao} ({time_slot[0]}-{time_slot[1]}): Status {response.status_code}")

    # Combining all individual DataFrames into a single master DataFrame
    if list_for_arrivals_df:
        return pd.concat(list_for_arrivals_df, ignore_index=True)
    else:
        return pd.DataFrame()


# Fetch flights using the dynamically retrieved ICAO list
flights_df = tomorrows_flight_arrivals(icao_list, api_key)

# Display the first few rows of the generated flights DataFrame
flights_df.head()

,arrival_airport_icao,flight_number,airline,arrival_time,arrival_terminal,departure_city,departure_airport_icao,data_retrieved_on
0,EDDB,WT 4006,Wasaya,2026-09-29 05:00,NaN,Paris,NaN,2026-09-28 20:49:12
1,EDDB,XQ 966,Sun Express,2026-09-29 06:05,1,İzmir,LTBJ,2026-09-28 20:49:12
2,EDDB,XQ 1766,Sun Express,2026-09-29 06:45,1,Gaziantep,LTAJ,2026-09-28 20:49:12
3,EDDB,W4 3109,Aero Services Executive,2026-09-29 07:00,2,Bucharest,LROP,2026-09-28 20:49:12
4,EDDB,UA 962,United,2026-09-29 07:15,1,Newark,KEWR,2026-09-28 20:49:12


dataframe is now ready to be sent to sql but the problem is we are missing foreign and primary key.
In this step we will add city_id and city_name columns to this dataframe from sql

In [ ]:
# Reopen MySQL connection to Gans and pull city_id and city_name

import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

# Re-establish MySQL connection to Gans database
db_connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="Gans"
)

cursor = db_connection.cursor(buffered=True)

# Pulling city_id and city_name from the MySQL database
cursor.execute("SELECT city_id, city_name FROM cities;")
city_records = cursor.fetchall()

# Creating a dictionary mapping city_name to city_id (e.g., {'Berlin': 1, 'Hamburg': 2, 'Munich': 3})
city_id_map = {row[1]: row[0] for row in city_records}

# Function to query ICAO codes and map them directly to MySQL city_id
def get_airport_icaos_with_ids(city_id_map, api_key):
    icao_to_city_id = {}
    icao_list = []
    
    headers = {
        "X-RapidAPI-Key": api_key,
        "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
    }

    for city_name, city_id in city_id_map.items():
        url = "https://aerodatabox.p.rapidapi.com/airports/search/term"
        querystring = {"q": city_name, "limit": "1"}

        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            if "items" in data and len(data["items"]) > 0:
                icao_code = data["items"][0].get("icao")
                if icao_code:
                    icao_list.append(icao_code)
                    icao_to_city_id[icao_code] = city_id
                    print(f"Mapped '{city_name}' (ID: {city_id}) -> ICAO: {icao_code}")

    return icao_list, icao_to_city_id


# Execute step to generate ICAO list and city_id mapping
icao_list, icao_to_city_id = get_airport_icaos_with_ids(city_id_map, api_key)
print("\nGenerated ICAO to City ID Map:", icao_to_city_id)

Mapped 'Berlin' (ID: 1) -> ICAO: EDDB
Mapped 'Hamburg' (ID: 2) -> ICAO: EDDH
Mapped 'Munich' (ID: 3) -> ICAO: EDDM

Generated ICAO to City ID Map: {'EDDB': 1, 'EDDH': 2, 'EDDM': 3}


In [15]:
# Function to get tomorrow's flight arrivals with mapped city_id
def tomorrows_flight_arrivals_with_city_id(icao_to_city_id, api_key):
    berlin_timezone = timezone('Europe/Berlin')
    today = datetime.now(berlin_timezone).date()
    tomorrow = today + timedelta(days=1)

    list_for_arrivals_df = []

    for icao, city_id in icao_to_city_id.items():
        times = [["00:00", "11:59"], ["12:00", "23:59"]]

        for time_slot in times:
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time_slot[0]}/{tomorrow}T{time_slot[1]}"
            querystring = {"direction": "Arrival", "withCancelled": "false"}

            headers = {
                "X-RapidAPI-Key": api_key,
                "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
            }

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:
                flights_resp = response.json()

                if "arrivals" in flights_resp and flights_resp["arrivals"]:
                    arrivals_df = pd.json_normalize(flights_resp["arrivals"])[
                        [
                            "number", 
                            "airline.name", 
                            "movement.scheduledTime.local", 
                            "movement.terminal", 
                            "movement.airport.name", 
                            "movement.airport.icao"
                        ]
                    ]

                    # Correct column renaming (Original API field -> Target field)
                    arrivals_df = arrivals_df.rename(columns={
                        "number": "flight_number", 
                        "airline.name": "airline", 
                        "movement.scheduledTime.local": "arrival_time", 
                        "movement.terminal": "arrival_terminal", 
                        "movement.airport.name": "departure_city", 
                        "movement.airport.icao": "departure_airport_icao"
                    })

                    arrivals_df["arrival_airport_icao"] = icao
                    arrivals_df["city_id"] = city_id
                    arrivals_df["data_retrieved_on"] = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")

                    arrivals_df["arrival_time"] = arrivals_df["arrival_time"].str.split("+").str[0]

                    list_for_arrivals_df.append(arrivals_df)

    if list_for_arrivals_df:
        return pd.concat(list_for_arrivals_df, ignore_index=True)
    else:
        return pd.DataFrame()


# Execute the complete flight pipeline
flights_df = tomorrows_flight_arrivals_with_city_id(icao_to_city_id, api_key)
flights_df.head()

,flight_number,airline,arrival_time,arrival_terminal,departure_city,departure_airport_icao,arrival_airport_icao,city_id,data_retrieved_on
0,WT 4006,Wasaya,2026-09-29 05:00,NaN,Paris,NaN,EDDB,1,2026-09-28 20:53:27
1,XQ 966,Sun Express,2026-09-29 06:05,1,İzmir,LTBJ,EDDB,1,2026-09-28 20:53:27
2,XQ 1766,Sun Express,2026-09-29 06:45,1,Gaziantep,LTAJ,EDDB,1,2026-09-28 20:53:27
3,W4 3109,Aero Services Executive,2026-09-29 07:00,2,Bucharest,LROP,EDDB,1,2026-09-28 20:53:27
4,UA 962,United,2026-09-29 07:15,1,Newark,KEWR,EDDB,1,2026-09-28 20:53:27


In [ ]:


load_dotenv()

# Database credentials
user = "root"
password = os.getenv("MYSQL_PASSWORD")
host = "127.0.0.1"
port = "3306"
database = "Gans"

# Create SQLAlchemy Engine
engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}")

# Send flight data to MySQL flights table
flights_df.to_sql('flights', con=engine, if_exists='append', index=False)

print("All flight data successfully pushed to MySQL Gans database!")

All flight data successfully pushed to MySQL Gans database!
